# 🧠 Spatial-Symbolic Transformer for ARC

## Overview
This notebook implements a **position-aware transformer architecture** specifically designed for ARC matrix reasoning:
- **Input**: Matrix values with explicit (x, y) positions
- **Architecture**: Spatial-Symbolic Transformer with self-attention
- **Goal**: Native spatial reasoning for discrete symbolic matrices

## Key Innovation
Unlike CNN-based encoders that treat matrices as images, this architecture:
- Converts matrices to sequences of (value, x, y) tokens
- Uses self-attention to learn spatial relationships
- Handles variable matrix sizes naturally
- Provides explicit positional reasoning capabilities

In [ ]:
# Import required libraries
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import math
import time
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Set matplotlib style
plt.style.use('default')
sns.set_palette("husl")

print("\n🚀 Spatial-Symbolic Transformer Environment Ready!")

In [ ]:
# Load ARC dataset (same as before)
def load_arc_data():
    """Load ARC training data from Kaggle input directory"""
    
    # Kaggle data paths
    train_challenges_path = '/kaggle/input/arc-prize-2025/arc-agi_training_challenges.json'
    train_solutions_path = '/kaggle/input/arc-prize-2025/arc-agi_training_solutions.json'
    
    try:
        # Load training challenges
        with open(train_challenges_path, 'r') as f:
            challenges = json.load(f)
        
        # Load training solutions
        with open(train_solutions_path, 'r') as f:
            solutions = json.load(f)
        
        print(f"✅ Loaded {len(challenges)} training tasks")
        print(f"✅ Loaded {len(solutions)} training solutions")
        
        return challenges, solutions
    
    except FileNotFoundError as e:
        print(f"❌ ARC data files not found: {e}")
        print("📁 Expected paths:")
        print(f"   • {train_challenges_path}")
        print(f"   • {train_solutions_path}")
        print("\n💡 Solutions:")
        print("   1. Upload this notebook to Kaggle with ARC Prize 2025 dataset")
        print("   2. Or modify the paths above to point to your local ARC data files")
        print("   3. Download ARC data from: https://www.kaggle.com/competitions/arc-prize-2025/data")
        
        raise FileNotFoundError("ARC dataset not found. Please check file paths or run on Kaggle.")

# Load the data
print("🔄 Loading ARC dataset...")
challenges, solutions = load_arc_data()

# Quick dataset verification
sample_task_id = list(challenges.keys())[0]
sample_task = challenges[sample_task_id]

print(f"\n📋 Sample Task ID: {sample_task_id}")
print(f"📊 Number of training examples: {len(sample_task['train'])}")
print(f"🧪 Number of test examples: {len(sample_task['test'])}")

# Show first training example structure
first_example = sample_task['train'][0]
input_grid = np.array(first_example['input'])
output_grid = np.array(first_example['output'])

print(f"\n🔍 First training example analysis:")
print(f"   Input shape: {input_grid.shape}")
print(f"   Output shape: {output_grid.shape}")
print(f"   Input unique values: {np.unique(input_grid)}")
print(f"   Output unique values: {np.unique(output_grid)}")

print(f"\n✅ Data loading successful! Ready for spatial-symbolic processing.")

In [ ]:
# Step 3: Matrix-to-Sequence Converter - Core Innovation
class MatrixToSequenceConverter:
    """
    Converts ARC matrices to sequences of (value, x, y) tokens for transformer processing.
    This is the key innovation that makes position explicit rather than implicit.
    """
    
    def __init__(self, normalize_positions=True):
        self.normalize_positions = normalize_positions
    
    def matrix_to_sequence(self, matrix):
        """
        Convert matrix to sequence of position-aware tokens.
        
        Args:
            matrix: numpy array of shape (height, width) with values 0-9
            
        Returns:
            tokens: list of (value, x, y) tuples
            metadata: dict with original dimensions and stats
        """
        height, width = matrix.shape
        tokens = []
        
        # Convert each cell to (value, x, y) token
        for y in range(height):
            for x in range(width):
                value = int(matrix[y, x])
                
                if self.normalize_positions:
                    # Normalize positions to [0, 1] range for better learning
                    norm_x = x / max(width - 1, 1)  # Avoid division by zero
                    norm_y = y / max(height - 1, 1)
                    tokens.append((value, norm_x, norm_y))
                else:
                    # Use absolute positions
                    tokens.append((value, x, y))
        
        # Metadata for reconstruction and analysis
        metadata = {
            'height': height,
            'width': width,
            'num_tokens': len(tokens),
            'unique_values': len(np.unique(matrix)),
            'total_cells': height * width
        }
        
        return tokens, metadata
    
    def sequence_to_matrix(self, tokens, target_height, target_width):
        """
        Convert sequence back to matrix (for reconstruction testing).
        
        Args:
            tokens: list of (value, x, y) tuples
            target_height, target_width: desired output dimensions
            
        Returns:
            matrix: reconstructed numpy array
        """
        matrix = np.zeros((target_height, target_width), dtype=int)
        
        for value, x, y in tokens:
            if self.normalize_positions:
                # Denormalize positions
                abs_x = int(round(x * max(target_width - 1, 1)))
                abs_y = int(round(y * max(target_height - 1, 1)))
            else:
                abs_x, abs_y = int(x), int(y)
            
            # Ensure coordinates are within bounds
            if 0 <= abs_y < target_height and 0 <= abs_x < target_width:
                matrix[abs_y, abs_x] = int(value)
        
        return matrix
    
    def visualize_conversion(self, matrix, tokens, metadata):
        """Visualize the matrix-to-sequence conversion process."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # Original matrix
        im1 = ax1.imshow(matrix, cmap='tab10', vmin=0, vmax=9)
        ax1.set_title(f'Original Matrix ({metadata["height"]}x{metadata["width"]})')
        ax1.set_xlabel('X coordinate')
        ax1.set_ylabel('Y coordinate')
        
        # Add grid and labels
        ax1.set_xticks(range(metadata['width']))
        ax1.set_yticks(range(metadata['height']))
        ax1.grid(True, alpha=0.3)
        
        # Token sequence visualization
        values = [token[0] for token in tokens]
        x_coords = [token[1] for token in tokens]
        y_coords = [token[2] for token in tokens]
        
        scatter = ax2.scatter(x_coords, y_coords, c=values, cmap='tab10', 
                            vmin=0, vmax=9, s=100, alpha=0.8)
        ax2.set_title(f'Token Sequence ({len(tokens)} tokens)')
        ax2.set_xlabel('Normalized X' if self.normalize_positions else 'X coordinate')
        ax2.set_ylabel('Normalized Y' if self.normalize_positions else 'Y coordinate')
        
        # Add colorbar
        plt.colorbar(scatter, ax=ax2, label='Value')
        
        plt.tight_layout()
        plt.show()
        
        return fig

# Test the Matrix-to-Sequence Converter
print("🔧 Testing Matrix-to-Sequence Conversion...")

# Initialize converter
converter = MatrixToSequenceConverter(normalize_positions=True)

# Test with the sample matrix we loaded
test_matrix = input_grid  # Use the first example from ARC data
print(f"\n📊 Testing with matrix shape: {test_matrix.shape}")
print(f"Matrix content:\n{test_matrix}")

# Convert to sequence
tokens, metadata = converter.matrix_to_sequence(test_matrix)

print(f"\n🎯 Conversion Results:")
print(f"   • Original matrix: {metadata['height']}x{metadata['width']} = {metadata['total_cells']} cells")
print(f"   • Token sequence: {metadata['num_tokens']} tokens")
print(f"   • Unique values: {metadata['unique_values']}")

# Show first few tokens
print(f"\n🔍 First 10 tokens (value, norm_x, norm_y):")
for i, token in enumerate(tokens[:10]):
    value, x, y = token
    print(f"   Token {i}: value={value}, x={x:.3f}, y={y:.3f}")

# Test reconstruction
reconstructed = converter.sequence_to_matrix(tokens, metadata['height'], metadata['width'])
reconstruction_accurate = np.array_equal(test_matrix, reconstructed)

print(f"\n✅ Reconstruction Test:")
print(f"   • Original == Reconstructed: {reconstruction_accurate}")
if reconstruction_accurate:
    print("   🎉 Perfect round-trip conversion!")
else:
    print("   ❌ Reconstruction error - need to debug")

# Visualize the conversion
print(f"\n📈 Visualizing conversion process...")
converter.visualize_conversion(test_matrix, tokens, metadata)